In [ ]:
import os
import datetime
import sys
from random import choice
from glob import glob

from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

sys.path.insert(1, "/home/xilinx/qick-qoc/board/")

from programs.qdac import *
from programs.randomized_benchmarking2_with_single_shot_readout import *
from programs.res_spec import *

%matplotlib notebook

## LOAD FIRMWARE

In [ ]:
# Load bitstream with custom overlay
soccfg = QickSoc(bitfile="/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/qick_size_mod_2024MAY08.bit", external_clk=True)

In [ ]:
qdac = QDAC_II()

## EXPERIMENT CONSTANTS

In [ ]:
# QICK
DAC_OUTCLOCK_RATE   = 6881.280e6
DAC_OUTCLOCK_PERIOD = 1/DAC_OUTCLOCK_RATE  # RFSoC DAC Sampling Interval from QICK config
ADC_OFFSET          = 275           # Time-of-flight calibration
RELAX_DELAY         = 1000.0

# Resonator
READOUT_RESONATOR_GAIN = 4500   # Power to resonator
READOUT_RESONATOR_FREQ = 311.03 # Resonator frequency [MHz]
READOUT_PULSE_LENGTH   = 10.0     # us2cycles value (needs calibration)

# FLUX TUNING
FT_QDAC_CHANNEL   = 3
FT_NUM_AVERAGES   = 4000
FT_QUBIT_GAIN     = 1000      # Power to qubit
FT_PROBE_LENGTH   = 1e-6   # s
FT_FREQ           = 224.704e6 # Hz
#
FT_FLUX_TESTRANGE     =  0.08
FT_FLUX_TESTNUMPOINTS =  26
#
RB_NUM_SHOTS         = 512
RB_QUBIT_FREQ        = 224.704e6  # Qubit frequency [Hz]
RB_QUBIT_PERIOD      = 1./RB_QUBIT_FREQ

In [ ]:
# RANOMIZED BENCHMARKING
RB_Xd2_SAMPLERATE      = 100*DAC_OUTCLOCK_RATE
RB_Xd2_NAMES           = ['n40c'] #["n50","n40c","n200"]
RB_Xd2_GAINS           = [1280] #[1285,1280,395]
RB_Xd2_PULSES          = [Xd2Gen.getXd2Pulse("/home/xilinx/data/rb_2024OCT/pronto_{}_2024OCT14.csv".format(name)) for name in RB_Xd2_NAMES]
RB_Xd2_PULSE_DURATIONS = [len(pulse)/RB_Xd2_SAMPLERATE for pulse in RB_Xd2_PULSES]

In [ ]:
(2**19/DAC_OUTCLOCK_RATE - 12e-6)/np.asarray(RB_Xd2_PULSE_DURATIONS)*.9

In [ ]:
RB_Xd2_MAX_NUM_PULSES  = [300] #[250,300,65]
RB_GATES_TESTPOINTS  = [np.geomspace(3, ii, 15, dtype=int) for ii in RB_Xd2_MAX_NUM_PULSES]

## PHASE CONFIGURATION

In [ ]:
PHASE_CONFIG_NUM_AVERAGES = 1000  # number of averages
PHASE_CONFIG_RELAX_DELAY = 10

hw_cfg = {"res_ch": 1, "qubit_ch": 0, "storage_ch": 0}  # No JPA channel
readout_cfg = {
    "readout_length": soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
    # "f_res": 99.775 + 0.18,  # [MHz]
    "res_phase": 0,
    "adc_trig_offset": 275,  # [Clock ticks]
    "frequency" : READOUT_RESONATOR_FREQ,
    "res_gain" : READOUT_RESONATOR_GAIN,
}

expt_cfg = {
    "reps": PHASE_CONFIG_NUM_AVERAGES,
    "relax_delay": PHASE_CONFIG_RELAX_DELAY,
    "start": 0,
    "step": 0,
    "expts": 1}
config = {**hw_cfg, **readout_cfg, **expt_cfg}  # combine configs

In [ ]:
phase_pts = np.linspace(0,360,361)

avg_i0 = np.empty([phase_pts.shape[0]])
avg_q0 = np.empty([phase_pts.shape[0]])
result = np.empty([phase_pts.shape[0]])

for p_idx, p in enumerate(tqdm(phase_pts)):
    config["res_phase"] = p
    rspec = SingleToneSpectroscopyProgram(soccfg, config)
    avgi, avgq = rspec.acquire(soccfg, load_pulses=True)
    avg_i0[p_idx] = avgi[0][0]
    avg_q0[p_idx] = avgq[0][0]

In [ ]:
# THE LAMEST VIDEO GAME EVER, MATCH THE ORANGE AND BLUE LINES TO WHERE THE SINUSOIDS CROSS (IS SPECIFIC TO THE SYSTEM...)
plt.figure()
plt.plot(phase_pts, avg_i0)
plt.plot(phase_pts, avg_q0)
plt.axhline(0)
plt.axhline(0.67,color='b')
plt.axhline(0,color='r')
plt.axhline(-0.67,color='orange')

In [ ]:
READOUT_RES_PHASE = 285 # degrees

## FLUX TUNING FUNCTIONS

In [ ]:
def init_flux_test_pulse(FT_FREQ):
    fluxtestpulsets = np.arange(0., FT_PROBE_LENGTH, DAC_OUTCLOCK_PERIOD)
    fluxtestpulse   = Xd2Gen.getGaussianSinusoidPulse(fluxtestpulsets, FT_FREQ, FT_PROBE_LENGTH/4)
    return fluxtestpulse

def run_flux_test(prev_flux): 
    results = []
    fluxes  = np.linspace(prev_flux-FT_FLUX_TESTRANGE/2, prev_flux+FT_FLUX_TESTRANGE/2, FT_FLUX_TESTNUMPOINTS)
    for flux in tqdm(fluxes, leave=False):
        qdac.set_voltage(FT_QDAC_CHANNEL,flux,dwell=0.1)
        #
        cfg    = genFluxTuningConfigs(FLUX_TEST_PULSE)
        rbprog = RandomizedBenchmarking2OriginalProgram(soccfg, cfg)
        avgi, avgq = rbprog.acquire(soccfg, load_pulses=True)
        #
        results.append(avgi[0][0])
    return fluxes, results

def gaussian(fluxes, b, c, amp, offset):
    f = amp*np.exp(-(fluxes-b)**2/(2*c**2))+offset
    return f

def get_optimal_flux_fit(fluxes, results):
    popt, pcov = curve_fit(gaussian, fluxes, results, maxfev=10000)
    return popt

def run_flux_calibration(currentoptimalflux):
    fluxes, results = run_flux_test(currentoptimalflux)
    popt = get_optimal_flux_fit(fluxes, results)
    new_optimal_flux = np.round(popt[0], 5)
    qdac.set_voltage(FT_QDAC_CHANNEL,new_optimal_flux,dwell=0.1)
    return fluxes, results, new_optimal_flux

## EXPERIMENT CONFIGS

In [ ]:
def genRandomizedBenchmarkingConfigs(arbitrarypulse, gain):
    qubit_cfg = {
        "qubit_ch"         : 0, # 4 in QICK supplied build
        "qubit_pulse_name" : "arbitrary_pulse",
        "qubit_i_data"     : (2**15-2)*arbitrarypulse,
        "qubit_q_data"     : np.zeros_like(arbitrarypulse),
        "qubit_gain"       : gain,
    }
    readout_cfg = {
        "res_ch"             : 1, # 6 in QICK supplied build
        "res_freq"           : READOUT_RESONATOR_FREQ,  # Resonator frequency [MHz]
        "res_phase"          : READOUT_RES_PHASE,
        "res_gain"           : READOUT_RESONATOR_GAIN,  # Power to resonator
        "res_readout_length" : soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
        "adc_trig_offset"    : ADC_OFFSET,  # [Clock ticks]
        "delta_t"            : 0,
    }
    expt_cfg = {
        "reps"        : RB_NUM_SHOTS,
        "rounds"      : 1,
        "relax_delay" : RELAX_DELAY,
    }
    cfg = {**readout_cfg, **qubit_cfg, **expt_cfg}  # combine configs
    return cfg

def genFluxTuningConfigs(arbitrarypulse):
    qubit_cfg = {
        "qubit_ch"         : 0, # 4 in QICK supplied build
        "qubit_pulse_name" : "arbitrary_pulse",
        "qubit_i_data"     : (2**15-2)*arbitrarypulse,
        "qubit_q_data"     : np.zeros_like(arbitrarypulse),
        "qubit_gain"       : FT_QUBIT_GAIN,
    }
    readout_cfg = {
        "res_ch"             : 1, # 6 in QICK supplied build
        "res_freq"           : READOUT_RESONATOR_FREQ,  # Resonator frequency [MHz]
        "res_phase"          : READOUT_RES_PHASE,
        "res_gain"           : READOUT_RESONATOR_GAIN,  # Power to resonator
        "res_readout_length" : soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
        "adc_trig_offset"    : ADC_OFFSET,  # [Clock ticks]
    }
    expt_cfg = {
        "reps"        : FT_NUM_AVERAGES,
        "rounds"      : 1,
        "relax_delay" : RELAX_DELAY,
    }
    cfg = {**readout_cfg, **qubit_cfg, **expt_cfg}  # combine configs
    return cfg

## FLUX TEST

In [ ]:
currentoptimalflux = -.03

In [ ]:
FLUX_TEST_PULSE = init_flux_test_pulse(FT_FREQ)
fluxes, flux_test_results, currentoptimalflux = run_flux_calibration(currentoptimalflux)

In [ ]:
plt.figure()
plt.plot(fluxes, flux_test_results)
print(currentoptimalflux)

## RUN QUICK X/2 TEST

In [ ]:
num_xd2_gates    = np.arange(3,8,1)
xd2_gate_results = np.zeros(len(num_xd2_gates))

for ii, num in enumerate(tqdm(num_xd2_gates)):
    test_gates = ['X/2']*num
    PulseGen.genArbitraryPulseStart(RB_Xd2_PULSES[0], RB_Xd2_PULSE_DURATIONS[0], RB_QUBIT_PERIOD, DAC_OUTCLOCK_PERIOD)
    PulseGen.genArbitraryPulseAddGates(test_gates)
    arbitrarypulsets, arbitrarypulse = PulseGen.genArbitraryPulseGetOutput()
    #
    cfg          = genRandomizedBenchmarkingConfigs(arbitrarypulse, RB_Xd2_GAINS[0])
    rbprog       = RandomizedBenchmarking2OriginalProgram(soccfg, cfg)
    result       = rbprog.acquire(soccfg, load_pulses=True)
    xd2_gate_results[ii] += (np.sqrt(result[0][1][0]**2 + result[1][1][0]**2 + result[0][0][0]**2 + result[1][0][0]**2))

In [ ]:
plt.figure()
plt.plot(num_xd2_gates, xd2_gate_results, 'o-')
#There isn't a lot of averaging, so it's not going to look amazing...

## RANDOMIZED BENCHMARKING

In [ ]:
SAVE_FOLDER = "/home/xilinx/data/rb_2024OCT/rb2024OCT28"

In [ ]:
N = len(RB_Xd2_NAMES)
fluxtestpulsets = np.arange(0., 10e-6, DAC_OUTCLOCK_PERIOD)
warmup = Xd2Gen.getGaussianSinusoidPulse(fluxtestpulsets, 200e6, 12e-6)

for ii in tqdm(range(201)):
    pulse               = RB_Xd2_PULSES[ii%N]
    pulseduration       = RB_Xd2_PULSE_DURATIONS[ii%N]
    pulsegain           = RB_Xd2_GAINS[ii%N]
    pulsename           = RB_Xd2_NAMES[ii%N]
    numgatestestpoints  = RB_GATES_TESTPOINTS[ii%N]
    #
    # FLUX TUNING
    timestamp             = datetime.datetime.now().replace(microsecond=0).isoformat()
    fluxes, flux_test_results, currentoptimalflux = run_flux_calibration(currentoptimalflux)
    ft_filename = os.path.join(SAVE_FOLDER, timestamp.replace(":", "") + "_ft_{}_results_it{}".format(pulsename, ii) + ".npz")
    np.savez(ft_filename, fluxes=fluxes, flux_test_results=flux_test_results, currentoptimalflux=currentoptimalflux)
    #
    # RANDOMIZED BENCHMARKING
    print("ITERATION:",ii,"FLUX:",currentoptimalflux,"PULSE_TYPE:",pulsename)
    resultsarray     = [None]*len(numgatestestpoints)
    timestamp        = datetime.datetime.now().replace(microsecond=0).isoformat()
    randomgatesarray = [None]*len(numgatestestpoints)
    inversetypearray = [None]*len(numgatestestpoints)
    inversegatearray = [None]*len(numgatestestpoints)
    for jj, num_gates in enumerate(numgatestestpoints):
        randomgates = CliffordGates.genRandomCliffordGates(num_gates)
        if(choice([0, 1])==0):
            inversegate = CliffordGates.getInverseCliffordGate(CliffordGates.getProductOfCliffordGates(randomgates))
            inversetype = 'I'
        else:
            inversegate = CliffordGates.getXInverseCliffordGate(CliffordGates.getProductOfCliffordGates(randomgates))
            inversetype = 'X'
        #
        randomgatesarray[jj] = randomgates
        inversetypearray[jj] = inversetype
        inversegatearray[jj] = inversegate
        #
        PulseGen.genArbitraryPulseStart(pulse, pulseduration, RB_QUBIT_PERIOD, DAC_OUTCLOCK_PERIOD)
        for randomgate in randomgates:
            PulseGen.genArbitraryPulseAddGates(CliffordGates.getCliffordGateDecomposition(randomgate))
        PulseGen.genArbitraryPulseAddGates(CliffordGates.getCliffordGateDecomposition(inversegate))
        arbitrarypulsets, arbitrarypulse = PulseGen.genArbitraryPulseGetOutput()
        #
        warmup_plus_arbitrarypulse = np.concatenate((warmup,arbitrarypulse))
        #
        cfg              = genRandomizedBenchmarkingConfigs(arbitrarypulse, pulsegain)
        rbprog           = RandomizedBenchmarking2SingleShotReadoutProgram(soccfg, cfg)
        result           = rbprog.acquire(soccfg, load_pulses=True)
        resultsarray[jj] = result
    rb_filename = os.path.join(SAVE_FOLDER, timestamp.replace(":", "") + "_rb_{}_results_it{}".format(pulsename, ii) + ".npz")
    np.savez(rb_filename, numgatestestpoints=numgatestestpoints, randomgatesarray=randomgatesarray, inversetypearray=inversetypearray, inversegatearray=inversegatearray, resultsarray=resultsarray, currentoptimalflux=currentoptimalflux)

In [ ]:
parameters_filename = os.path.join(SAVE_FOLDER, timestamp.replace(":", "") + "parameters.npz")
np.savez(
    parameters_filename,
    DAC_OUTCLOCK_RATE = DAC_OUTCLOCK_RATE,
    DAC_OUTCLOCK_PERIOD = DAC_OUTCLOCK_PERIOD,
    ADC_OFFSET = ADC_OFFSET,
    RELAX_DELAY = RELAX_DELAY,
    READOUT_RESONATOR_GAIN = READOUT_RESONATOR_GAIN,
    READOUT_RESONATOR_FREQ = READOUT_RESONATOR_FREQ,
    READOUT_PULSE_LENGTH = READOUT_PULSE_LENGTH,
    RB_NUM_SHOTS = RB_NUM_SHOTS,
    RB_QUBIT_FREQ = RB_QUBIT_FREQ,
    RB_QUBIT_PERIOD = RB_QUBIT_PERIOD,
    RB_Xd2_SAMPLERATE = RB_Xd2_SAMPLERATE,
    RB_Xd2_NAMES = RB_Xd2_NAMES,
    RB_Xd2_GAINS = RB_Xd2_GAINS,
    RB_Xd2_PULSES = RB_Xd2_PULSES,
    RB_Xd2_PULSE_DURATIONS = RB_Xd2_PULSE_DURATIONS,
    RB_Xd2_MAX_NUM_PULSES = RB_Xd2_MAX_NUM_PULSES,
    RB_GATES_TESTPOINTS = RB_GATES_TESTPOINTS,
    # FLUX TUNING
    FT_QDAC_CHANNEL = FT_QDAC_CHANNEL,
    FT_NUM_AVERAGES = FT_NUM_AVERAGES,
    FT_QUBIT_GAIN = FT_QUBIT_GAIN, # Power to qubit
    FT_PROBE_LENGTH = FT_PROBE_LENGTH, # s
    FT_FREQ = FT_FREQ,
    FT_FLUX_TESTRANGE = FT_FLUX_TESTRANGE,
    FT_FLUX_TESTNUMPOINTS = FT_FLUX_TESTNUMPOINTS,
)

In [ ]:
qdac.close()